# Scop3P

A comprehensive database of human phosphosites within their full context. Scop3P integrates sequences (UniProtKB/Swiss-Prot), structures (PDB), and uniformly reprocessed phosphoproteomics data (PRIDE) to annotate all known human phosphosites. 

Scop3P, available at https://iomics.ugent.be/scop3p, presents a unique resource for visualization and analysis of phosphosites and for understanding of phosphosite structure–function relationships.

Please cite: https://doi.org/10.1021/acs.jproteome.0c00306

# Scop3P-Structural and biophysical visualization framework

This notebook renders analysis in multiple tabs:


>1. Fetch PTMs using Scop3P API and all single-site UniProt PTM features
>2. Fetch disease variants using UniProt API 
>3. 3D visualization (Mapping PTMs to experimental PDB and AlphaFold structures)
>4. Predict Biophysical properties using Bio2byte tools and map onto 3D structures (single or multi panel)
>5. Residue Interaction Network (RIN) constructions and visualization (We move from 3D to 2.5D to get more insights on local residue interactions)
>6. Structure alignment (Align two structures to see the structural similarity/difference)



In [4]:
import requests, tempfile,json
import pandas as pd 
from b2bTools import SingleSeq, constants
import py3Dmol
import os
import nglview as nv


/home/paddy/venvs/ptm/lib/python3.12/site-packages/nglview/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [5]:
import tempfile

state = {
    "acc": None,
    "ptm_results": None,      # keep if you use it
    "ptm_json": None,         # add from state
    "ptm_table": None,
    "variants": None,
    "variants_df": None,      # add from state (or choose one naming)
    "sequence": None,
    "bio2byte_raw": None,
    "dynamic_properties": None,
    "rin_pdb_path": None,
    "af_path": None,
    "rin_html": None,
    "protein_info": None,
    "ptm_source_label": None,
    "workdir": tempfile.mkdtemp(prefix="scop3p_session_")
}


In [6]:
def fetch_protein_modifications(accession):
    """
    Fetch protein modifications from Scop3P for a given UniProt accession.

    Scop3P currently mainly covers human phosphoproteins. For non-human proteins,
    or proteins not present in Scop3P, this function can return None/empty data.
    """
    BASE_URL = "https://iomics.ugent.be/scop3p/api/modifications"
    url = f"{BASE_URL}?accession={accession}"
    headers = {"accept": "application/json"}
    response = requests.get(url, headers=headers, timeout=60)
    if response.status_code == 200:
        return response.json()
    return None


_AA1_TO_AA3 = {
    "A": "ALA", "R": "ARG", "N": "ASN", "D": "ASP", "C": "CYS",
    "Q": "GLN", "E": "GLU", "G": "GLY", "H": "HIS", "I": "ILE",
    "L": "LEU", "K": "LYS", "M": "MET", "F": "PHE", "P": "PRO",
    "S": "SER", "T": "THR", "W": "TRP", "Y": "TYR", "V": "VAL",
    "U": "SEC", "O": "PYL",
}


def _residue_from_uniprot_feature(sequence, position, description=""):
    """Return a three-letter residue code for a UniProt PTM feature."""
    desc = (description or "").lower()

    # Prefer explicit residue names in the UniProt PTM description.
    if "phosphoserine" in desc:
        return "SER"
    if "phosphothreonine" in desc:
        return "THR"
    if "phosphotyrosine" in desc:
        return "TYR"

    # Otherwise infer from the sequence position when available.
    try:
        pos = int(position)
        if sequence and 1 <= pos <= len(sequence):
            return _AA1_TO_AA3.get(sequence[pos - 1].upper(), sequence[pos - 1].upper())
    except Exception:
        pass

    return ""


def _format_uniprot_evidence(evidences):
    """Condense UniProt feature evidence into evidence codes and literature references."""
    codes, refs = [], []
    for ev in evidences or []:
        code = ev.get("code")
        if code:
            codes.append(code)
        src = ev.get("source") or {}
        name = src.get("name")
        sid = src.get("id")
        if name and sid:
            refs.append(f"{name}:{sid}")
        elif sid:
            refs.append(str(sid))

    return "; ".join(dict.fromkeys(codes)), "; ".join(dict.fromkeys(refs))


def fetch_uniprot_ptms(accession: str) -> pd.DataFrame:
    """
    Fetch all single-site PTM features from the EBI/UniProt Proteins API.

    This uses categories=PTM without restricting feature types, so it can include
    phosphorylation, acetylation, methylation, glycosylation, lipidation and other
    UniProt PTM annotations when present. Only single-residue features are retained,
    meaning begin == end, because the rest of the app maps PTMs to residue positions.
    The UniProt description is trimmed at the first semicolon, for example
    "Phosphotyrosine; by autocatalysis" becomes "Phosphotyrosine".
    """
    url = f"https://www.ebi.ac.uk/proteins/api/features/{accession}"
    params = [("categories", "PTM")]
    headers = {"Accept": "application/json"}
    r = requests.get(url, headers=headers, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()

    sequence = data.get("sequence", "") or ""
    rows = []
    for feat in data.get("features", []) or []:
        if feat.get("category") != "PTM":
            continue

        begin = feat.get("begin")
        end = feat.get("end")
        if begin is None or end is None or str(begin) != str(end):
            continue

        try:
            position = int(begin)
        except Exception:
            continue

        full_desc = feat.get("description") or feat.get("type") or "PTM"
        clean_name = str(full_desc).split(";", 1)[0].strip()
        evidence, reference = _format_uniprot_evidence(feat.get("evidences"))

        rows.append({
            "ACC_ID": accession,
            "residue": _residue_from_uniprot_feature(sequence, position, clean_name),
            "name": clean_name,
            "evidence": evidence,
            "position": position,
            "source": "UniProt",
            "reference": reference,
            "functionalScore": pd.NA,
            "specificSinglyPhosphorylated": pd.NA,
            "feature_type": feat.get("type"),
        })

    cols = [
        "ACC_ID", "residue", "name", "evidence", "position", "source",
        "reference", "functionalScore", "specificSinglyPhosphorylated", "feature_type",
    ]
    return pd.DataFrame(rows, columns=cols)



# ----------------------------
# UniProt protein summary helpers for Tab 1
# ----------------------------
def _safe_html(value):
    """Small HTML escaping helper without requiring an extra import in the notebook."""
    if value is None:
        return ""
    return (str(value)
            .replace("&", "&amp;")
            .replace("<", "&lt;")
            .replace(">", "&gt;")
            .replace('"', "&quot;"))


def _get_uniprot_recommended_name(data):
    pdsc = data.get("proteinDescription") or {}
    rec = pdsc.get("recommendedName") or {}
    full = rec.get("fullName") or {}
    if full.get("value"):
        return full.get("value")

    sub = pdsc.get("submissionNames") or []
    if sub:
        full = (sub[0] or {}).get("fullName") or {}
        if full.get("value"):
            return full.get("value")
    return data.get("uniProtkbId") or data.get("primaryAccession") or ""


def _get_uniprot_gene_names(data):
    genes = []
    for g in data.get("genes", []) or []:
        gn = (g.get("geneName") or {}).get("value")
        if gn:
            genes.append(gn)
    return ", ".join(dict.fromkeys(genes))


def _extract_comment_text(comment):
    texts = []
    for t in comment.get("texts", []) or []:
        val = t.get("value")
        if val:
            texts.append(val)
    return " ".join(texts).strip()


def _get_subcellular_locations(data):
    vals = []
    for c in data.get("comments", []) or []:
        if c.get("commentType") != "SUBCELLULAR LOCATION":
            continue
        for loc in c.get("subcellularLocations", []) or []:
            location = ((loc.get("location") or {}).get("value") or "").strip()
            topology = ((loc.get("topology") or {}).get("value") or "").strip()
            orientation = ((loc.get("orientation") or {}).get("value") or "").strip()
            parts = [p for p in [location, topology, orientation] if p]
            if parts:
                vals.append("; ".join(parts))
        txt = _extract_comment_text(c)
        if txt:
            vals.append(txt)
    return "; ".join(dict.fromkeys(vals))


def _get_function_annotation(data, max_chars=260):
    for c in data.get("comments", []) or []:
        if c.get("commentType") == "FUNCTION":
            txt = _extract_comment_text(c)
            if txt:
                return txt[:max_chars].rstrip() + ("..." if len(txt) > max_chars else "")
    return ""


def fetch_uniprot_protein_info(accession: str) -> dict:
    """Fetch compact UniProtKB protein metadata for the Tab 1 information card."""
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    data = r.json()

    organism = ((data.get("organism") or {}).get("scientificName") or "")
    seq = data.get("sequence") or {}
    keywords = [k.get("name") for k in data.get("keywords", []) or [] if k.get("name")]

    return {
        "accession": data.get("primaryAccession") or accession,
        "entry_name": data.get("uniProtkbId") or "",
        "protein_name": _get_uniprot_recommended_name(data),
        "gene": _get_uniprot_gene_names(data),
        "organism": organism,
        "length": seq.get("length") or len(seq.get("value", "") or ""),
        # UniProt REST commonly returns values such as
        # "UniProtKB reviewed (Swiss-Prot)" or "UniProtKB unreviewed (TrEMBL)".
        # Check for reviewed while explicitly excluding unreviewed.
        "reviewed": ("reviewed" in str(data.get("entryType", "")).lower()
                     and "unreviewed" not in str(data.get("entryType", "")).lower()),
        "subcellular_location": _get_subcellular_locations(data),
        "function": _get_function_annotation(data),
        "keywords": ", ".join(keywords[:8]),
    }


def render_protein_info_html(info: dict, ptm_label: str = "") -> str:
    """Render a compact protein information card for Voila."""
    if not info:
        return "<div style='color:#666;'>Protein information will appear after setting/fetching a UniProt accession.</div>"

    reviewed = "Reviewed" if info.get("reviewed") else "Unreviewed"
    subcell = info.get("subcellular_location") or "Not annotated in UniProt"
    function = info.get("function") or "Not shown"
    keywords = info.get("keywords") or ""
    ptm_line = f"<div><b>PTM source:</b> {_safe_html(ptm_label)}</div>" if ptm_label else ""

    return f"""
    <div style="border:1px solid #d9e2ec; border-radius:8px; padding:10px 12px; background:#f8fbff; margin:6px 0 10px 0;">
      <div style="font-size:15px; margin-bottom:4px;"><b>{_safe_html(info.get('protein_name'))}</b></div>
      <div style="display:grid; grid-template-columns: repeat(2, minmax(220px, 1fr)); gap:4px 18px; font-size:13px;">
        <div><b>Accession:</b> {_safe_html(info.get('accession'))} {_safe_html(info.get('entry_name'))}</div>
        <div><b>Gene:</b> {_safe_html(info.get('gene') or 'N/A')}</div>
        <div><b>Organism:</b> {_safe_html(info.get('organism') or 'N/A')}</div>
        <div><b>Length:</b> {_safe_html(info.get('length') or 'N/A')} aa | {_safe_html(reviewed)}</div>
        {ptm_line}
        <div><b>Keywords:</b> {_safe_html(keywords or 'N/A')}</div>
      </div>
      <div style="font-size:13px; margin-top:6px;"><b>Subcellular location:</b> {_safe_html(subcell)}</div>
      <div style="font-size:13px; margin-top:4px;"><b>Function:</b> {_safe_html(function)}</div>
    </div>
    """


In [7]:
import re
def get_modification_table(modifications, accession=None, source="Scop3P"):
    """
    Convert Scop3P modification records into the PTM table schema used by the app.
    """
    base_cols = [
        "ACC_ID", "residue", "name", "evidence", "position", "source",
        "reference", "functionalScore", "specificSinglyPhosphorylated", "feature_type",
    ]

    if not modifications:
        return pd.DataFrame(columns=base_cols)

    df = pd.DataFrame(modifications)

    # Keep legacy Scop3P columns, but make the function robust if any field is absent.
    for col in ["residue", "name", "evidence", "position", "source", "reference", "functionalScore", "specificSinglyPhosphorylated"]:
        if col not in df.columns:
            df[col] = pd.NA

    if "ACC_ID" not in df.columns:
        df["ACC_ID"] = accession
    if "feature_type" not in df.columns:
        df["feature_type"] = "Scop3P"

    # Do not overwrite a meaningful Scop3P source, but fill blanks.
    df["source"] = df["source"].fillna(source)
    df.loc[df["source"].astype(str).str.strip().eq(""), "source"] = source

    return df[base_cols]




def _join_unique_values(*values, sep="; "):
    """Join non-empty scalar/list values while preserving first-seen order."""
    seen, out = set(), []
    for value in values:
        if value is None or (hasattr(pd, "isna") and not isinstance(value, (list, tuple, set)) and pd.isna(value)):
            continue
        if isinstance(value, (list, tuple, set)):
            parts = value
        else:
            # Preserve comma-separated Scop3P PubMed style and semicolon-separated UniProt style.
            parts = re.split(r"\s*[;,]\s*", str(value))
        for part in parts:
            part = str(part).strip()
            if not part or part.lower() in {"nan", "<na>", "none"}:
                continue
            if part not in seen:
                seen.add(part)
                out.append(part)
    return sep.join(out)


def merge_scop3p_uniprot_ptms(scop3p_tbl, uniprot_tbl):
    """
    Merge Scop3P and UniProt PTMs without duplicating the same residue-position site.

    If a site exists in both sources, the displayed row follows Scop3P terminology
    and Scop3P source fields (for example: name='phosphorylation', evidence='Experimental',
    source='UP', feature_type='Scop3P'). UniProt references/evidence are folded into
    the Scop3P row where useful. UniProt-only PTMs are retained as UniProt rows.
    """
    base_cols = [
        "ACC_ID", "residue", "name", "evidence", "position", "source",
        "reference", "functionalScore", "specificSinglyPhosphorylated", "feature_type",
    ]

    scop3p_tbl = scop3p_tbl.copy() if scop3p_tbl is not None else pd.DataFrame(columns=base_cols)
    uniprot_tbl = uniprot_tbl.copy() if uniprot_tbl is not None else pd.DataFrame(columns=base_cols)

    for df in (scop3p_tbl, uniprot_tbl):
        for col in base_cols:
            if col not in df.columns:
                df[col] = pd.NA
        if not df.empty:
            df["position"] = pd.to_numeric(df["position"], errors="coerce").astype("Int64")
            df["ACC_ID"] = df["ACC_ID"].astype(str).str.strip()
            df["residue"] = df["residue"].astype(str).str.strip().str.upper()

    if scop3p_tbl.empty:
        out = uniprot_tbl[base_cols].copy()
        return out.sort_values(["position", "source", "name"], na_position="last").reset_index(drop=True)
    if uniprot_tbl.empty:
        out = scop3p_tbl[base_cols].copy()
        return out.sort_values(["position", "source", "name"], na_position="last").reset_index(drop=True)

    # Use accession + residue + position as the biological site identity.
    key_cols = ["ACC_ID", "residue", "position"]
    scop3p_tbl["_site_key"] = list(map(tuple, scop3p_tbl[key_cols].astype(str).values))
    uniprot_tbl["_site_key"] = list(map(tuple, uniprot_tbl[key_cols].astype(str).values))

    scop3p_by_key = {k: idx for idx, k in enumerate(scop3p_tbl["_site_key"].tolist())}
    merged_rows = scop3p_tbl.copy()

    # Fold UniProt evidence/reference into matching Scop3P rows, but keep Scop3P naming/source style.
    for _, urow in uniprot_tbl.iterrows():
        key = urow["_site_key"]
        if key not in scop3p_by_key:
            continue
        idx = scop3p_by_key[key]
        sref = merged_rows.at[idx, "reference"]
        uref = urow.get("reference", pd.NA)
        merged_rows.at[idx, "reference"] = _join_unique_values(sref, uref, sep=",")

        # Keep the compact Scop3P evidence label when present; otherwise borrow UniProt evidence.
        sev = merged_rows.at[idx, "evidence"]
        if pd.isna(sev) or str(sev).strip() == "":
            merged_rows.at[idx, "evidence"] = urow.get("evidence", pd.NA)

    uniprot_only = uniprot_tbl[~uniprot_tbl["_site_key"].isin(set(scop3p_by_key.keys()))].copy()
    out = pd.concat([merged_rows, uniprot_only], ignore_index=True)
    out = out.drop(columns=[c for c in ["_site_key"] if c in out.columns])
    out = out[base_cols]
    out = out.drop_duplicates(subset=key_cols + ["name", "source", "feature_type"], keep="first")
    return out.sort_values(["position", "source", "name"], na_position="last").reset_index(drop=True)


In [8]:
import requests
import pandas as pd

def fetch_uniprot_variants_disease(accession: str) -> pd.DataFrame:
    """Fetch UniProt (EBI proteins API) variants with disease association for a UniProt accession."""
    url = f"https://www.ebi.ac.uk/proteins/api/variation/{accession}"
    headers = {"Accept": "application/json"}
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()
    data = r.json()

    rows = []
    for feat in data.get("features", []):
        if feat.get("type") != "VARIANT":
            continue
        for assoc in feat.get("association", []):
            if assoc.get("disease") is not True:
                continue
            begin = feat.get("begin")
            try:
                pos = int(begin) if begin is not None else None
            except Exception:
                pos = None
            rows.append({
                "ACC_ID": accession,
                "position": pos,
                "WT": feat.get("wildType"),
                "MT": feat.get("mutatedType"),
                "consequence": feat.get("consequenceType"),
                "disease_name": assoc.get("name"),
            })

    return pd.DataFrame(rows)


In [9]:
import py3Dmol

def display_local_pdb_3D(modification_table, accession):
    view = py3Dmol.view(width=700, height=500)
    view.addModel(open(accession + '.pdb', 'r').read(), 'pdb')

    view.setStyle({}, {'cartoon': {'color': 'silver'}})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.35, 'color': 'white'}, {})

    # --- Color phosphosites 
    for _, row in modification_table.iterrows():
        position = str(row['position'])

        # Normalize residue label to avoid mismatches
        residue = str(row['residue']).strip()  # removes trailing spaces etc.

        if residue == 'TYR':
            color = '#2CA02C'
        elif residue == 'SER':
            color = '#1F77B4'
        elif residue == 'THR':
            color = '#FF7F0E'
        else:
            color = '#7B241C'

        sel = {'resi': position}  # add {'chain': row['chain']} if needed

        view.addStyle(sel, {'stick': {'color': color}})
        view.addStyle(sel, {'sphere': {'color': color, 'radius': 0.9}})

    # --- Hover for ALL amino acids (all atoms) ---
    view.setHoverable(
        {}, True,
        """
        function(atom, viewer, event, container) {
            if(!atom.label) {
                atom.label = viewer.addLabel(
                    atom.resn + " " + atom.resi + (atom.chain ? (" : " + atom.chain) : ""),
                    {position: atom, backgroundColor: 'mintcream', fontColor: 'black'}
                );
            }
        }
        """,
        """
        function(atom, viewer) {
            if(atom.label) {
                viewer.removeLabel(atom.label);
                delete atom.label;
            }
        }
        """
    )

    view.zoomTo()
    view.render()
    return view


In [10]:
def display_ptm_3D(modification_table, pdb_id, chain=None):
    view = py3Dmol.view(query=f"pdb:{pdb_id}")

    # Protein context
    view.setStyle({}, {'cartoon': {'color': 'skyblue'}})

    # Global surface (NO hover expected here)
    view.addSurface(py3Dmol.VDW, {'opacity': 0.6, 'color': 'white'}, {})

    # ---- Colored modified residues (ATOMS) ----
    for _, row in modification_table.iterrows():
        position = str(row['position'])
        residue  = str(row['residue']).strip()

        if residue == 'TYR':
            color = '#2CA02C'
        elif residue == 'SER':
            color = '#1F77B4'
        elif residue == 'THR':
            color = '#FF7F0E'
        else:
            color = '#7B241C'

        sel = {'resi': position}
        if chain:
            sel['chain'] = chain

        # ATOMS → hover works
        view.addStyle(sel, {'stick':  {'color': color}})
        view.addStyle(sel, {'sphere': {'color': color, 'radius': 0.9}})

    # ---- Hover for ALL amino acids ----
    view.setHoverable(
        {}, True,
        """
        function(atom, viewer, event, container) {
            if (!atom.label) {
                atom.label = viewer.addLabel(
                    atom.resn + " " + atom.resi + (atom.chain ? (" : " + atom.chain) : ""),
                    {position: atom, backgroundColor: 'mintcream', fontColor: 'black'}
                );
            }
        }
        """,
        """
        function(atom, viewer) {
            if (atom.label) {
                viewer.removeLabel(atom.label);
                delete atom.label;
            }
        }
        """
    )

    view.zoomTo()
    view.render()
    return view


In [11]:
import os
import requests

def download_alphafold_pdb(uniprot_acc: str, outdir: str) -> str:
    """
    Downloads AlphaFold DB PDB for a UniProt accession.
    Returns local file path.
    """
    os.makedirs(outdir, exist_ok=True)
    # AFDB file naming convention
    url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_acc}-F1-model_v6.pdb"
    out_path = os.path.join(outdir, f"AF-{uniprot_acc}-F1-model_v6.pdb")

    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        f.write(r.content)

    return out_path


In [12]:
def fetch_sequence_aminoacids(accession):
    BASE_URL = f"http://uniprot.org/uniprotkb/{accession}.fasta"
    url = f'{BASE_URL}?accession={accession}'
    response = requests.get(url)
    if response.status_code == 200:
        raw_fasta_sequence = response.content.decode("utf-8")
    else:
        raw_fasta_sequence = ""
    
    lines = raw_fasta_sequence.split('\n')
    protein_id = str(lines[0])
    amino_acids = "".join([str(l) for l in lines[1:]])
    
    return protein_id, amino_acids

In [13]:
def predict_biophysical_features(accession, sequence):

    with tempfile.NamedTemporaryFile(prefix="seq_", suffix=".fasta", mode="w") as fp:
        fp.write(f">{accession}\n{sequence}\n")
        fp.flush()
        fp.seek(0)
        
        pred = SingleSeq(fp.name).predict(tools=[constants.TOOL_DYNAMINE, constants.TOOL_DISOMINE, constants.TOOL_EFOLDMINE]).get_all_predictions()
    
    return pred


In [14]:
import colorsys


def pseudocolor(minval, maxval,val):
    """ Convert predicted values min.....max in range Green...Yellow..RED 
        The colors correspond to Red and Green in the HSV colorspace
    """
    minval,maxval=float(minval),float(maxval)
    h = (float(maxval-val) / (maxval-minval)) * 120
    r, g, b = colorsys.hsv_to_rgb(h/360, 1., 1.)
    rgb=map(lambda x: int(255 * x), (r, g, b))
    rgb=tuple(rgb)
    rgb='0x%02x%02x%02x' % rgb
    return rgb

In [15]:
def remap(df):
    BDcolor,EFcolor,DOcolor={},{},{}
    seqpos=0
    min_BD,max_BD=min(df.backbone),max(df.backbone)
    min_DO,max_DO=min(df.disoMine),max(df.disoMine)
    min_EF,max_EF=min(df.earlyFolding),max(df.earlyFolding)
    
    for index, row in df.iterrows():
        seqpos+=1
        BDrescol=pseudocolor(min_BD,max_BD,float(row.backbone))
        DOrescol=pseudocolor(min_EF,max_EF,float(row.disoMine))
        EFrescol=pseudocolor(min_EF,max_EF,float(row.earlyFolding))
        BDcolor[seqpos]=BDrescol
        DOcolor[seqpos]=DOrescol
        EFcolor[seqpos]=EFrescol
        
    return BDcolor,EFcolor,DOcolor
        
        

In [16]:
def display_b2b_3D(dynamic_properties, pdb_path: str):
    BDcolor, EFcolor, DOcolor = remap(dynamic_properties)
    modpos = modification_table.position.tolist()

    view = py3Dmol.view(viewergrid=(2,2))
    with open(pdb_path, "r") as f:
        view.addModel(f.read(), "pdb")

    # IMPORTANT: setStyle(selection, style)
    view.setStyle({}, {'cartoon': {'colorscheme': {'prop':'b','gradient':'rwb','min':0.0,'max':100.0}}}, viewer=(0,0))
    view.setStyle({}, {'cartoon': {'colorscheme': {'prop':'resi','map':BDcolor}}}, viewer=(0,1))
    view.setStyle({}, {'cartoon': {'colorscheme': {'prop':'resi','map':DOcolor}}}, viewer=(1,0))
    view.setStyle({}, {'cartoon': {'colorscheme': {'prop':'resi','map':EFcolor}}}, viewer=(1,1))

    # Surface highlight + pickable overlay on mod residues
    for mod in modpos:
        m = str(mod)
        sel = {'resi': m}

        view.addSurface(py3Dmol.VDW, {'opacity': 1.0}, sel, viewer=(0,0))
        view.addSurface(py3Dmol.VDW, {'opacity': 1.0, 'color': BDcolor[mod]}, sel, viewer=(0,1))
        view.addSurface(py3Dmol.VDW, {'opacity': 1.0, 'color': DOcolor[mod]}, sel, viewer=(1,0))
        view.addSurface(py3Dmol.VDW, {'opacity': 1.0, 'color': EFcolor[mod]}, sel, viewer=(1,1))

        # MAKE IT PICKABLE: opacity must be > 0
        for panel in [(0,0), (0,1), (1,0), (1,1)]:
            view.addStyle(sel, {'sphere': {'radius': 0.8, 'opacity': 0.15}}, viewer=panel)
            # optional: stick helps pickability even more
            # view.addStyle(sel, {'stick': {'opacity': 0.15}}, viewer=panel)

    # Background + hover everywhere (per panel)
    for panel in [(0,0), (0,1), (1,0), (1,1)]:
        view.setBackgroundColor('white', viewer=panel)

        view.setHoverable(
            {},  # hover everywhere
            True,
            """
            function(atom, viewer, event, container) {
                if (!atom.label) {
                    atom.label = viewer.addLabel(
                        atom.resn + " " + atom.resi + (atom.chain ? (" : " + atom.chain) : ""),
                        {position: atom, backgroundColor: 'mintcream', fontColor:'black'}
                    );
                }
            }
            """,
            """
            function(atom, viewer) {
                if (atom.label) {
                    viewer.removeLabel(atom.label);
                    delete atom.label;
                }
            }
            """,
            viewer=panel
        )

    view.zoomTo()
    view.render()
    return view


In [17]:
import numpy as np
import networkx as nx
from scipy.spatial import KDTree
from Bio.PDB import PDBParser

def build_geometry_graph_from_pdb(pdb_path, chain="A", cutoff=8.0, atom_name="CA"):
    """
    Build a residue interaction network from a PDB file using CA (or CB fallback) distances.
    Nodes: residue positions (ints)
    Edges: if distance <= cutoff, with attributes distance, weight=1/distance, resistance=distance
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("af", pdb_path)

    # Use first model
    model = next(structure.get_models())

    # Pick chain (AlphaFold is usually 'A')
    if chain not in model:
        chain_obj = next(model.get_chains())
        chain = chain_obj.id  # fallback
    else:
        chain_obj = model[chain]


    coords = []
    meta = []

    for res in chain_obj:
        # standard residues only
        if res.id[0] != " ":
            continue

        resi = int(res.id[1])
        resn = res.resname

        # choose atom
        atom = None
        if atom_name in res:
            atom = res[atom_name]
        elif atom_name == "CB" and "CA" in res:
            atom = res["CA"]
        elif atom_name == "CA":
            # CA required; skip if missing
            continue
        else:
            # fallback to CA if present
            atom = res["CA"] if "CA" in res else None

        if atom is None:
            continue

        coords.append(atom.coord.astype(float))
        meta.append({"Chain": chain, "Residue": resi, "ResName": resn})

    coords = np.asarray(coords, dtype=float)
    if len(coords) == 0:
        raise ValueError("No residue coordinates found. Check chain/atom_name.")

    nodes = [(m["Chain"], int(m["Residue"])) for m in meta]
    tree = KDTree(coords)

    G = nx.Graph(layer=f"geometry:{atom_name}_cut{cutoff}", chain=chain, pdb=pdb_path)

    for n, m in zip(nodes, meta):
        G.add_node(n, **m)

    for i in range(len(nodes)):
        idxs = tree.query_ball_point(coords[i], cutoff)
        for j in idxs:
            if j <= i:
                continue
            d = float(np.linalg.norm(coords[i] - coords[j]))
            w = 1.0 / max(d, 1e-6)
            G.add_edge(nodes[i], nodes[j], weight=w, distance=d, resistance=1.0 / max(w, 1e-9))

    return G, meta


In [18]:
from pyvis.network import Network

def nx_rin_to_pyvis_default(
    G,
    ptm_positions=None,
    mutation_positions=None,
    out_html="rin_pyvis.html",
    height="600px",
    width="100%",
    default_color="#B0B0B0",   # light grey
    ptm_color="#1f77b4",       # blue
    mut_color="#d62728",       # red
    both_color="#9467bd",      # purple
    node_size=30,
    ptm_size=35,
    mut_size=35,
    both_size=40,
    select_menu=True,
    filter_menu=False
):
    ptm_set = set(int(x) for x in (ptm_positions or []))
    mut_set = set(int(x) for x in (mutation_positions or []))

    net = Network(
        height=height,
        width=width,
        directed=False,
        notebook=True,
        cdn_resources="in_line",
        select_menu=select_menu,
        filter_menu=filter_menu
    )

    net.set_options("""
    {
      "groups": {
        "PTM": {
          "color": {
            "background": "#d62728",
            "border": "#d62728",
            "highlight": { "background": "#d62728", "border": "#d62728" },
            "hover":     { "background": "#d62728", "border": "#d62728" }
          }
        },
        "Mutation": {
          "color": {
            "background": "#1f77b4",
            "border": "#1f77b4",
            "highlight": { "background": "#1f77b4", "border": "#1f77b4" },
            "hover":     { "background": "#1f77b4", "border": "#1f77b4" }
          }
        },
        "PTM+Mutation": {
          "color": {
            "background": "#2ca02c",
            "border": "#2ca02c",
            "highlight": { "background": "#2ca02c", "border": "#2ca02c" },
            "hover":     { "background": "#2ca02c", "border": "#2ca02c" }
          }
        },
        "Other": {
          "color": {
            "background": "#9FA8B0",
            "border": "#9FA8B0",
            "highlight": { "background": "#9FA8B0", "border": "#9FA8B0" },
            "hover":     { "background": "#9FA8B0", "border": "#9FA8B0" }
          }
        }
      }
    }
    """)




    # ---- Nodes ----
    for (ch, resi), attrs in G.nodes(data=True):
        resi = int(resi)
        resn = attrs.get("ResName", "")

        is_ptm = resi in ptm_set
        is_mut = resi in mut_set

        if is_ptm and is_mut:
            bg = both_color
            size = both_size
            group = "PTM+Mutation"
        elif is_mut:
            bg = mut_color
            size = mut_size
            group = "Mutation"
        elif is_ptm:
            bg = ptm_color
            size = ptm_size
            group = "PTM"
        else:
            bg = default_color
            size = node_size
            group = "Other"

        node_id = f"{ch}:{resi}"

        net.add_node(
            node_id,
            label=f"{resn} {resi}" if resn else str(resi),
            title=f"{resn}:{ch}:{resi}:{group}",
            color={
                "background": bg,
                "border": "#333333",
                "highlight": {"background": bg, "border": "#000000"},
                "hover": {"background": bg, "border": "#000000"}
            },
            size=size,
            group=group,
            font={"size": 12}
        )

    # ---- Edges ----
    for (ch1, r1), (ch2, r2), eattrs in G.edges(data=True):
        a = f"{ch1}:{int(r1)}"
        b = f"{ch2}:{int(r2)}"
        dist = eattrs.get("distance", None)

        net.add_edge(
            a, b,
            color="#A9A9A9",
            title=f"distance: {dist:.2f} Å" if dist is not None else ""
        )

    html = net.generate_html()
    with open(out_html, "w", encoding="utf-8") as f:
        f.write(html)
    
    return out_html


In [19]:
import os
import subprocess
import tempfile
from Bio.PDB import PDBParser, PDBIO, Select
import nglview as nv
import ipywidgets as widgets
from IPython.display import display, clear_output

def save_upload(upload_widget, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    v = upload_widget.value
    if not v:
        raise ValueError("No file uploaded")

    # Newer ipywidgets: tuple/list of dicts
    if isinstance(v, (tuple, list)):
        fileinfo = v[0]
        name = fileinfo.get("name", "upload.pdb")
        content = fileinfo["content"]

    # Older ipywidgets: dict name -> fileinfo
    elif isinstance(v, dict):
        name, fileinfo = next(iter(v.items()))
        content = fileinfo["content"]

    else:
        raise TypeError(f"Unexpected upload_widget.value type: {type(v)}")

    path = os.path.join(out_dir, name)
    with open(path, "wb") as f:
        f.write(content)
    return path



def chain_range_from_pdb(pdb_path, chain_id):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("X", pdb_path)
    residues = [
        res.id[1]
        for model in structure
        for chain in model
        if chain.id == chain_id
        for res in chain
        if res.id[0] == " "
    ]
    if not residues:
        raise ValueError(f"No residues found for chain {chain_id}")
    return min(residues), max(residues)


class ChainRangeSelect(Select):
    def __init__(self, chain_id, start, end):
        self.chain_id = chain_id
        self.start = start
        self.end = end

    def accept_chain(self, chain):
        return chain.id == self.chain_id

    def accept_residue(self, residue):
        r = residue.id[1]
        return (self.start <= r <= self.end)

def run_tmalign_write(pdb1, pdb2, out_dir, out_name):
    os.makedirs(out_dir, exist_ok=True)
    cmd = ["TM-align", os.path.abspath(pdb1), os.path.abspath(pdb2), "-o", out_name]
    res = subprocess.run(cmd, cwd=out_dir, capture_output=True, text=True, check=True)

    candidates = [
        os.path.join(out_dir, out_name),
        os.path.join(out_dir, out_name + ".pdb"),
        os.path.join(out_dir, "TM_sup.pdb"),
    ]
    out_pdb = next((c for c in candidates if os.path.exists(c)), None)
    if out_pdb is None:
        raise RuntimeError(f"No TM-align output found. Files: {os.listdir(out_dir)}")
    return out_pdb, res.stdout

import nglview as nv

def visualize_ngl(pdb_ref, pdb_aligned, selection="protein"):
    view = nv.NGLWidget()

    # CRITICAL: disable default rainbow reps
    view.add_component(pdb_ref, ext="pdb", defaultRepresentation=False)
    view.add_component(pdb_aligned, ext="pdb", defaultRepresentation=False)

    view.clear_representations()

    # Reference (blue, translucent)
    view.add_cartoon(
        component=0,
        selection=selection,
        colorScheme="uniform",
        colorValue="blue",
        opacity=0.7
    )

    # Aligned (red, solid)
    view.add_cartoon(
        component=1,
        selection=selection,
        colorScheme="uniform",
        colorValue="red",
        opacity=1.0
    )

    view.center()
    return view


In [20]:
upload1 = widgets.FileUpload(accept=".pdb", multiple=False, description="Upload PDB 1")
upload2 = widgets.FileUpload(accept=".pdb", multiple=False, description="Upload PDB 2")

chain1 = widgets.Text(value="A", description="Chain 1")
chain2 = widgets.Text(value="A", description="Chain 2")

start1 = widgets.IntText(description="Start 1")
end1   = widgets.IntText(description="End 1")
start2 = widgets.IntText(description="Start 2")
end2   = widgets.IntText(description="End 2")

btn_range = widgets.Button(description="Auto-fill ranges")
btn_run = widgets.Button(description="Align + Visualize", button_style="primary")

out = widgets.Output()
workdir = tempfile.mkdtemp(prefix="tmalign_upload_tool_")


In [21]:
# Original TM-align upload-only autofill callback replaced by enhanced dropdown-aware callback in the main app cell.

In [22]:
# Original TM-align upload-only run callback replaced by enhanced dropdown-aware callback in the main app cell.

In [23]:
tmalign_ui = widgets.VBox([
    widgets.HBox([upload1, upload2]),
    widgets.HBox([chain1, start1, end1]),
    widgets.HBox([chain2, start2, end2]),
    widgets.HBox([btn_range, btn_run]),
    out,
])


In [24]:
import os, tempfile, traceback, re
import ipywidgets as w
from IPython.display import display, clear_output, IFrame, HTML
import requests

# ----------------------------
# Scrollable df helper
# ----------------------------
def display_scrollable_df(df, max_height="420px", max_width="100%"):
    if df is None:
        return
    html_table = df.to_html(index=False, escape=False)
    css = f"""
    <style>
      .scroll-df-wrap {{
        width: 100%;
        max-width: {max_width};
        max-height: {max_height};
        overflow-x: auto;
        overflow-y: auto;
        border: 1px solid #e0e0e0;
        border-radius: 6px;
        box-sizing: border-box;
      }}
      .scroll-df-wrap table {{
        min-width: 100%;
        border-collapse: collapse;
        font-size: 13px;
      }}
      .scroll-df-wrap th,
      .scroll-df-wrap td {{
        padding: 6px 8px;
        border-bottom: 1px solid #eee;
        text-align: left;
        white-space: nowrap;
      }}
      .scroll-df-wrap thead th {{
        position: sticky;
        top: 0;
        background: #fafafa;
        z-index: 2;
        border-bottom: 1px solid #ddd;
      }}
    </style>
    """
    display(HTML(css + f"<div class='scroll-df-wrap'>{html_table}</div>"))

def _err(out: w.Output, e: Exception):
    with out:
        print("❌ Error:", e)
        traceback.print_exc()

def _need_acc(out: w.Output):
    if not state.get("acc"):
        with out:
            print("Set a UniProt accession first (top bar).")
        return True
    return False

def download_alphafold_pdb(accession: str, out_dir: str) -> str:
    """Download AlphaFold PDB (model_v6) to out_dir and return file path."""
    os.makedirs(out_dir, exist_ok=True)
    url = f"https://alphafold.ebi.ac.uk/files/AF-{accession}-F1-model_v6.pdb"
    fp = os.path.join(out_dir, f"AF-{accession}-F1-model_v6.pdb")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    with open(fp, "wb") as f:
        f.write(r.content)
    return fp

# ----------------------------
# Top bar: select protein
# ----------------------------
acc_input = w.Text(description="UniProt:", placeholder="e.g., P07949", layout=w.Layout(width="320px"))
btn_set = w.Button(description="Set protein", button_style="info")
lbl = w.HTML("<b>Current:</b> (not set)")
top_out = w.Output(layout={"border":"1px solid #ddd","padding":"6px"})

# ----------------------------
# UniProt PDB cross-references (for Tab 3/4 dropdowns)
# ----------------------------
def fetch_uniprot_pdb_xrefs(accession: str):
    """Fetch UniProtKB JSON and parse PDB IDs + chain->(start,end) residue ranges when available."""
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    data = r.json()

    refs = []
    for x in data.get("uniProtKBCrossReferences", []) or []:
        if x.get("database") != "PDB":
            continue
        pdb_id = x.get("id")
        props = {p.get("key"): p.get("value") for p in (x.get("properties") or []) if isinstance(p, dict)}
        chains_raw = props.get("Chains") or props.get("Chain") or ""
        chain_ranges = {}

        # Chains formats seen in UniProt:
        #  "A=1-200" or "A=1-200, B=5-150" or "A=10-50; 70-120" or "A/B=1-120"
        if chains_raw:
            parts = [p.strip() for p in re.split(r",\s*(?=[A-Za-z0-9/]+=)", chains_raw) if p.strip()]
            for part in parts:
                if "=" not in part:
                    continue
                lhs, rngs = part.split("=", 1)
                lhs = lhs.strip()
                rngs = rngs.strip()

                starts, ends = [], []
                for m in re.finditer(r"(\d+)\s*-\s*(\d+)", rngs):
                    starts.append(int(m.group(1)))
                    ends.append(int(m.group(2)))

                # chain part may be A or A/B
                chains = [c.strip().upper()[:1] for c in re.split(r"[\/\s]+", lhs) if c.strip()]
                if starts and ends:
                    for ch in chains:
                        chain_ranges[ch] = (min(starts), max(ends))
                else:
                    for ch in chains:
                        chain_ranges.setdefault(ch, None)

        refs.append({
            "pdb_id": pdb_id,
            "chain_ranges": chain_ranges,   # may include None for range n/a
            "raw_chains": chains_raw,
            "method": props.get("Method"),
            "resolution": props.get("Resolution"),
        })
    return refs

def on_set(_):
    with top_out:
        clear_output()
        acc = acc_input.value.strip()
        if not acc:
            print("Please enter a UniProt accession.")
            return

        state["acc"] = acc
        lbl.value = f"<b>Current:</b> {acc}"

        # Fetch compact UniProt protein metadata for the Tab 1 information card
        try:
            state["protein_info"] = fetch_uniprot_protein_info(acc)
            protein_info_box.value = render_protein_info_html(state["protein_info"], state.get("ptm_source_label"))
        except Exception as e:
            state["protein_info"] = None
            protein_info_box.value = f"<div style='color:#a66;'>Protein information fetch failed: {_safe_html(e)}</div>"

        # Fetch UniProt PDB cross-references for dropdowns (Tabs 3/4/5/6)
        try:
            state["uniprot_pdb_refs"] = fetch_uniprot_pdb_xrefs(acc)
        except Exception as e:
            state["uniprot_pdb_refs"] = []
            print(f"⚠️ UniProt PDB cross-ref fetch failed: {e}")

        print(f"✅ Protein set: {acc}")
        print(f"Session workdir: {state['workdir']}")

        # Refresh Tab 3 dropdowns if present
        try:
            _refresh_pdb_dropdowns()
        except Exception:
            pass

        # Refresh Tab 4 dropdowns if present
        try:
            _refresh_b2b_pdb_dropdowns()
        except Exception:
            pass

        # Refresh Tab 5 dropdowns if present
        try:
            _refresh_rin_pdb_dropdowns()
        except Exception:
            pass

        # Refresh Tab 6 dropdowns if present
        try:
            _refresh_tm_pdb_dropdowns()
        except Exception:
            pass

btn_set.on_click(on_set)
top = w.VBox([w.HBox([acc_input, btn_set, lbl]), top_out])

# ----------------------------
# TAB 1: PTMs (Scop3P + UniProt)
# ----------------------------
out_ptm = w.Output(layout={"border":"1px solid #ddd","padding":"6px"})
btn_fetch_ptm = w.Button(description="Fetch PTMs", button_style="warning")
btn_show_ptm = w.Button(description="Show table")
include_uniprot_ptm = w.Checkbox(
    value=False,
    description="Also include all UniProt PTMs",
    indent=False,
    layout=w.Layout(width="420px")
)
ptm_header = w.HTML("<h3>Scop3P PTMs</h3>")
protein_info_box = w.HTML(render_protein_info_html(None))

def fetch_ptm(_):
    try:
        with out_ptm:
            clear_output()
            if _need_acc(out_ptm):
                return
            print("Fetching PTMs...")

        acc = state["acc"]

        # Always try Scop3P first. This preserves the original human Scop3P behavior.
        scop3p_res = fetch_protein_modifications(acc)
        scop3p_mods = (scop3p_res or {}).get("modifications", [])
        scop3p_tbl = get_modification_table(scop3p_mods, accession=acc, source="Scop3P")

        # UniProt PTMs are optional when Scop3P data exists, and automatic fallback otherwise.
        use_uniprot = include_uniprot_ptm.value or scop3p_tbl.empty
        if use_uniprot:
            uniprot_tbl = fetch_uniprot_ptms(acc)
        else:
            uniprot_tbl = pd.DataFrame(columns=scop3p_tbl.columns)

        if not scop3p_tbl.empty and not uniprot_tbl.empty:
            ptm_label = "Scop3P + UniProt PTMs"
        elif not scop3p_tbl.empty:
            ptm_label = "Scop3P PTMs"
        elif not uniprot_tbl.empty:
            ptm_label = "UniProt PTMs"
        else:
            ptm_label = "No PTMs found"

        # Merge Scop3P + UniProt by biological site.
        # Scop3P wins for duplicate phosphosites, so names such as "phosphorylation",
        # source keywords such as "UP", and the Scop3P feature label are retained.
        tbl = merge_scop3p_uniprot_ptms(scop3p_tbl, uniprot_tbl)

        state["ptm_json"] = {
            "Scop3P": scop3p_res,
            "UniProt": {"enabled": bool(use_uniprot), "n_features": len(uniprot_tbl)},
        }
        state["ptm_table"] = tbl
        state["ptm_source_label"] = ptm_label
        globals()["modification_table"] = tbl  # legacy compatibility

        ptm_header.value = f"<h3>{_safe_html(ptm_label)}</h3>"
        protein_info_box.value = render_protein_info_html(state.get("protein_info"), ptm_label)

        with out_ptm:
            print(f"✅ Done. Scop3P PTMs: {len(scop3p_tbl)} | UniProt PTMs shown: {len(uniprot_tbl)} | Total shown: {len(tbl)}")
            if not include_uniprot_ptm.value and not scop3p_tbl.empty:
                print("Tip: enable 'Also include all UniProt PTMs' to append all single-site UniProt PTM annotations.")
            display_scrollable_df(tbl, max_height="420px", max_width="95vw")

    except Exception as e:
        _err(out_ptm, e)

def show_ptm(_):
    with out_ptm:
        clear_output()
        if state.get("ptm_table") is None:
            print("No PTM table yet. Click 'Fetch PTMs'.")
            return
        display_scrollable_df(state["ptm_table"], max_height="420px", max_width="95vw")

btn_fetch_ptm.on_click(fetch_ptm)
btn_show_ptm.on_click(show_ptm)
tab1 = w.VBox([
    ptm_header,
    protein_info_box,
    w.HTML("<p>Fetches Scop3P PTMs by default. Enable the UniProt option to append all single-site UniProt PTM annotations, or use UniProt automatically when Scop3P has no PTMs.</p>"),
    include_uniprot_ptm,
    w.HBox([btn_fetch_ptm, btn_show_ptm]),
    out_ptm
])

# ----------------------------
# TAB 2: Variants (UniProt/EBI API)
# ----------------------------
out_var = w.Output(layout={"border":"1px solid #ddd","padding":"6px"})
btn_fetch_var = w.Button(description="Fetch disease-associated variants", button_style="warning")

def fetch_var(_):
    try:
        with out_var:
            clear_output()
            if _need_acc(out_var):
                return
            print("Fetching variants...")

        df = fetch_uniprot_variants_disease(state["acc"])
        state["variants_df"] = df

        with out_var:
            print("✅ Done.")
            display_scrollable_df(df, max_height="420px", max_width="95vw")

    except Exception as e:
        _err(out_var, e)

btn_fetch_var.on_click(fetch_var)
tab2 = w.VBox([w.HTML("<h3>Disease-associated variants</h3>"), btn_fetch_var, out_var])

# ----------------------------
# TAB 3: 3D structure viewer (PDB + AlphaFold)  [NGLVIEW ONLY]
# ----------------------------
import nglview as nv

out_3d = w.Output(layout={"border":"1px solid #ddd","padding":"8px"})

structure_source = w.ToggleButtons(
    options=[("PDB", "pdb"), ("AlphaFold", "af")],
    value="pdb",
    description="Source:"
)

pdb_input   = w.Text(description="PDB:", placeholder="e.g., 2IVT", layout=w.Layout(width="240px"))
chain_input = w.Text(description="Chain:", placeholder="A (optional)", layout=w.Layout(width="200px"))

map_mode = w.Dropdown(
    description="Map:",
    options=[("PTMs", "ptm"), ("Mutations", "mut"), ("Both", "both")],
    value="both",
    layout=w.Layout(width="220px")
)

# Dropdowns populated from UniProt PDB cross-references
pdb_dd   = w.Dropdown(options=[("", "")], value="", description="UniProt PDB:", layout=w.Layout(width="420px"))
chain_dd = w.Dropdown(options=[("", "")], value="", description="Chain/range:", layout=w.Layout(width="420px"))
range_lbl = w.HTML("")

def _refresh_pdb_dropdowns():
    refs = state.get("uniprot_pdb_refs") or []
    pdb_ids = sorted({(r.get("pdb_id") or "").upper() for r in refs if r.get("pdb_id")})
    if not pdb_ids:
        pdb_dd.options = [("", "")]
        pdb_dd.value = ""
        chain_dd.options = [("", "")]
        chain_dd.value = ""
        range_lbl.value = "<i>No UniProt PDB cross-references found.</i>"
        return

    pdb_dd.options = [("", "")] + [(pid, pid) for pid in pdb_ids]
    if pdb_dd.value not in [v for _, v in pdb_dd.options]:
        pdb_dd.value = ""
    _update_chain_dropdown_from_pdb()

def _update_chain_dropdown_from_pdb(*_):
    pid = (pdb_dd.value or "").upper().strip()
    if not pid:
        chain_dd.options = [("", "")]
        chain_dd.value = ""
        range_lbl.value = ""
        return

    refs = state.get("uniprot_pdb_refs") or []
    cr = {}
    for r in refs:
        if (r.get("pdb_id") or "").upper() != pid:
            continue
        for ch, rng in (r.get("chain_ranges") or {}).items():
            cr[ch] = rng

    if not cr:
        chain_dd.options = [("", "")]
        chain_dd.value = ""
        range_lbl.value = "<i>No chain information available in UniProt for this PDB.</i>"
        return

    opts = [("", "")]
    for ch in sorted(cr.keys()):
        rng = cr[ch]
        if rng and len(rng) == 2:
            label = f"{ch} ({rng[0]}-{rng[1]})"
        else:
            label = f"{ch} (range n/a)"
        opts.append((label, ch))
    chain_dd.options = opts
    if chain_dd.value not in [v for _, v in opts]:
        chain_dd.value = ""

    _sync_textboxes_from_dropdowns()

def _sync_textboxes_from_dropdowns(*_):
    if pdb_dd.value:
        pdb_input.value = pdb_dd.value
    if chain_dd.value:
        chain_input.value = chain_dd.value

    # update range label if known
    pid = (pdb_dd.value or "").upper().strip()
    ch = (chain_dd.value or "").strip().upper()[:1] if chain_dd.value else ""
    if pid and ch:
        refs = state.get("uniprot_pdb_refs") or []
        for r in refs:
            if (r.get("pdb_id") or "").upper() != pid:
                continue
            rng = (r.get("chain_ranges") or {}).get(ch)
            if rng and len(rng) == 2:
                range_lbl.value = f"<b>UniProt range:</b> {ch} = {rng[0]}–{rng[1]}"
                return
        range_lbl.value = f"<b>UniProt range:</b> {ch} = n/a"
    else:
        range_lbl.value = ""

pdb_dd.observe(_update_chain_dropdown_from_pdb, names="value")
chain_dd.observe(_sync_textboxes_from_dropdowns, names="value")

btn_fetch_af = w.Button(description="Fetch AlphaFold", button_style="warning")
btn_show_3d  = w.Button(description="Show 3D", button_style="success")

PTM_COLOR = {"SER": "red", "THR": "green", "TYR": "orange"}

def _download_pdb_to_workdir(pdb_id: str, outdir: str) -> str:
    pdb_id = pdb_id.strip().lower()
    os.makedirs(outdir, exist_ok=True)
    outpath = os.path.join(outdir, f"{pdb_id}.pdb")
    if os.path.exists(outpath) and os.path.getsize(outpath) > 0:
        return outpath

    url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    with open(outpath, "wb") as f:
        f.write(r.content)
    return outpath

def _build_nglview_from_pdbfile(pdb_path: str):
    v = nv.show_file(pdb_path)
    v.camera = "orthographic"
    try:
        v._remote_call("setParameters", target="stage", kwargs={"tooltip": True})
    except Exception:
        pass
    return v

def _style_base_ngl(view, chain=None):
    try:
        view.clear_representations()
    except Exception:
        pass
    chain = (chain or "").strip()
    if chain:
        chain = chain[0].upper()
        sel = f"protein and chain {chain}"
    else:
        sel = "protein"
    view.add_representation("cartoon", selection=sel, color="grey")

def _collect_ptms_from_table(ptm_table):
    if ptm_table is None:
        return []
    pos_col = next((c for c in ["position","modpos","Position","MODPOS"] if c in ptm_table.columns), None)
    res_col = next((c for c in ["residue","modres","Residue","MODRES"] if c in ptm_table.columns), None)
    if pos_col is None or res_col is None:
        return []
    out = []
    for _, row in ptm_table[[pos_col, res_col]].dropna().iterrows():
        try:
            pos = int(row[pos_col])
        except Exception:
            continue
        res = str(row[res_col]).strip().upper()
        out.append((pos, res))
    return out

def _collect_variant_positions(variants_df):
    if variants_df is None or len(variants_df) == 0:
        return []
    if "position" not in variants_df.columns:
        return []
    out = []
    for v in variants_df["position"].dropna().tolist():
        try:
            out.append(int(v))
        except Exception:
            continue
    return sorted(set(out))

def _highlight_sites_ngl(view, ptm_table, variants_df=None, chain=None, mode="both"):
    """
    NOTE: This keeps your working behaviour (overlay). If you currently use
    "cartoon recolor only", swap this with your recolor-only version.
    """
    ptms = _collect_ptms_from_table(ptm_table)
    muts = set(_collect_variant_positions(variants_df))

    chain = (chain or "").strip()
    if chain:
        chain = chain[0].upper()

    ptm_pos_to_res = {int(pos): str(res).strip().upper() for pos, res in ptms if pos is not None}

    if mode == "ptm":
        positions = sorted(ptm_pos_to_res.keys())
    elif mode == "mut":
        positions = sorted(muts)
    else:
        positions = sorted(set(ptm_pos_to_res.keys()) | set(muts))

    if not positions:
        return

    for pos in positions:
        in_ptm = pos in ptm_pos_to_res
        in_mut = pos in muts

        if mode == "ptm":
            if not in_ptm:
                continue
            color = PTM_COLOR.get(ptm_pos_to_res[pos], "#7B241C")
        elif mode == "mut":
            if not in_mut:
                continue
            color = "red"
        else:
            if in_ptm and in_mut:
                color = "green"
            elif in_mut:
                color = "red"
            else:
                color = PTM_COLOR.get(ptm_pos_to_res[pos], "#7B241C")

        sel = f"{int(pos)}:{chain}" if chain else f"{int(pos)}"
        view.add_representation("spacefill", selection=sel, color=color, radius=0.9)

def on_fetch_af(_):
    try:
        with out_3d:
            clear_output()
            if not state.get("acc"):
                print("Set a UniProt accession first (header).")
                return
            print("Downloading AlphaFold model...")
            af_path = download_alphafold_pdb(state["acc"], out_dir=state["workdir"])
            state["af_path"] = af_path
            print("✅ Saved:", af_path)
    except Exception as e:
        with out_3d:
            print("❌ Failed to fetch AlphaFold:", e)

def on_show_3d(_):
    try:
        with out_3d:
            clear_output()

            ptm_table = state.get("ptm_table")

            # AlphaFold branch
            if structure_source.value == "af":
                af_path = state.get("af_path")
                if not af_path or not os.path.exists(af_path):
                    print("No AlphaFold model found. Click 'Fetch AlphaFold' first.")
                    return
                v = _build_nglview_from_pdbfile(af_path)
                _style_base_ngl(v)
                _highlight_sites_ngl(v, ptm_table, state.get("variants_df"), chain="A", mode=map_mode.value)
                display(v)
                return

            # PDB branch
            pdb = pdb_input.value.strip()
            if not pdb:
                print("Enter a PDB ID.")
                return

            ch = chain_input.value.strip() or None
            if ch:
                ch = ch[0].upper()

            pdb_path = _download_pdb_to_workdir(pdb, state["workdir"])
            v = _build_nglview_from_pdbfile(pdb_path)
            _style_base_ngl(v, chain=ch)
            _highlight_sites_ngl(v, ptm_table, state.get("variants_df"), chain=ch, mode=map_mode.value)
            display(v)

    except Exception as e:
        with out_3d:
            print("❌ Error:", e)

btn_fetch_af.on_click(on_fetch_af)
btn_show_3d.on_click(on_show_3d)

pdb_controls = w.VBox([
    w.HBox([pdb_input, chain_input]),
    w.HBox([pdb_dd, chain_dd]),
    range_lbl,
])

def _sync_tab3_visibility(*_):
    with out_3d:
        clear_output()

    if structure_source.value == "af":
        pdb_input.value = ""
        chain_input.value = ""
        try: pdb_dd.value = ""
        except Exception: pass
        try: chain_dd.value = ""
        except Exception: pass

        pdb_controls.layout.display = "none"
        btn_fetch_af.layout.display = ""
        with out_3d:
            print("AlphaFold selected. Click 'Fetch AlphaFold' (then 'Show 3D').")
    else:
        pdb_controls.layout.display = ""
        btn_fetch_af.layout.display = "none"
        with out_3d:
            print("PDB selected. Enter a PDB ID + (optional) chain, or pick from the UniProt dropdowns.")

structure_source.observe(_sync_tab3_visibility, names="value")
_sync_tab3_visibility()

tab3 = w.VBox([
    w.HTML("<h3>3D Structure Viewer</h3>"),
    w.HBox([structure_source, btn_fetch_af, btn_show_3d, map_mode]),
    pdb_controls,
    out_3d
])

# Initialize Tab 3 dropdowns (safe)
try:
    _refresh_pdb_dropdowns()
except Exception:
    pass

# ----------------------------
# TAB 4: Bio2Byte predictions (on-demand)
# ----------------------------
out_b2b = w.Output(layout={"border":"1px solid #ddd","padding":"6px"})
btn_fetch_seq = w.Button(description="Fetch sequence", button_style="warning")
btn_run_b2b = w.Button(description="Run predictions", button_style="danger")
btn_show_b2b = w.Button(description="Show prediction table", button_style="success")
btn_show_b2b3d = w.Button(description="3D panel (Bio2Byte colors)", button_style="")

b2b_metric = w.Dropdown(description="Color by:", options=[], layout=w.Layout(width="320px"))

viewer_mode = w.ToggleButtons(
    options=[("Single (NGL)", "ngl"), ("4-panel (py3Dmol)", "py3dmol")],
    value="ngl",
    description="Viewer:",
    layout=w.Layout(margin="0 0 0 10px")
)

# structure source
b2b_source = w.ToggleButtons(
    options=[("AlphaFold", "af"), ("PDB", "pdb")],
    value="af",
    description="Structure:",
    layout=w.Layout(width="360px")
)

# PDB entry + dropdowns
b2b_pdb_text  = w.Text(description="PDB ID:", placeholder="e.g. 6CER", layout=w.Layout(width="220px"))
b2b_chain_text = w.Text(description="Chain:", placeholder="A", layout=w.Layout(width="160px"))

b2b_pdb_dropdown   = w.Dropdown(description="UniProt PDB:", options=[""], layout=w.Layout(width="320px"))
b2b_chain_dropdown = w.Dropdown(description="Chain/range:", options=[("", "")], layout=w.Layout(width="380px"))

# ---- FIXED: Tab 4 dropdown population (show ALL chains, ranges if available) ----
def _refresh_b2b_pdb_dropdowns():
    """
    Populate Tab 4 PDB + chain/range dropdowns from state['uniprot_pdb_refs'].

    Shows ALL chains mentioned by UniProt.
    If range is known -> label "A (UniProt 10–220)" value "A|10|220"
    If range is n/a    -> label "A (range n/a)"       value "A||"
    """
    refs = state.get("uniprot_pdb_refs") or []

    pdb_to_chains = {}  # pid -> { chain -> (u0,u1) or None }

    for r in refs:
        pid = (r.get("pdb_id") or "").strip().upper()
        if not pid:
            continue
        chain_map = pdb_to_chains.setdefault(pid, {})

        cr = r.get("chain_ranges") or {}
        for ch, rng in cr.items():
            if not ch:
                continue
            ch = ch.strip().upper()[:1]
            if rng and isinstance(rng, (tuple, list)) and len(rng) == 2:
                try:
                    chain_map[ch] = (int(rng[0]), int(rng[1]))
                except Exception:
                    chain_map.setdefault(ch, None)
            else:
                chain_map.setdefault(ch, None)

        # Also include any chain letters in raw_chains even if range not parsed
        raw = (r.get("raw_chains") or "").strip()
        if raw:
            for part in [p.strip() for p in raw.split(",") if p.strip()]:
                if "=" not in part:
                    continue
                lhs = part.split("=", 1)[0].strip()
                for ch in re.split(r"[\/\s]+", lhs):
                    ch = (ch or "").strip().upper()
                    if not ch:
                        continue
                    ch = ch[:1]
                    chain_map.setdefault(ch, None)

    state["b2b_pdb_chain_map"] = pdb_to_chains

    pdb_ids = sorted(pdb_to_chains.keys())
    b2b_pdb_dropdown.options = [""] + pdb_ids

    if b2b_pdb_dropdown.value not in ([""] + pdb_ids):
        b2b_pdb_dropdown.value = ""

    _b2b_on_pdb_pick()

def _b2b_on_pdb_pick(change=None):
    pid = (b2b_pdb_dropdown.value or "").strip().upper()
    pdb_map = state.get("b2b_pdb_chain_map") or {}

    if not pid or pid not in pdb_map:
        b2b_chain_dropdown.options = [("", "")]
        b2b_chain_dropdown.value = ""
        return

    chain_map = pdb_map[pid]  # { 'A': (u0,u1) or None }

    opts = [("", "")]
    for ch in sorted(chain_map.keys()):
        rng = chain_map[ch]
        if rng and len(rng) == 2:
            u0, u1 = int(rng[0]), int(rng[1])
            opts.append((f"{ch} (UniProt {u0}–{u1})", f"{ch}|{u0}|{u1}"))
        else:
            opts.append((f"{ch} (range n/a)", f"{ch}||"))

    b2b_chain_dropdown.options = opts
    b2b_chain_dropdown.value = ""

def _b2b_on_chain_pick(change=None):
    val = b2b_chain_dropdown.value or ""
    if "|" in val:
        parts = (val.split("|") + ["", ""])[:3]
        ch = (parts[0] or "").strip().upper()[:1]
        if ch:
            b2b_chain_text.value = ch
        if b2b_pdb_dropdown.value:
            b2b_pdb_text.value = b2b_pdb_dropdown.value

b2b_pdb_dropdown.observe(_b2b_on_pdb_pick, names="value")
b2b_chain_dropdown.observe(_b2b_on_chain_pick, names="value")

# ---- end dropdown plumbing ----

import colorsys
import py3Dmol
import pandas as pd

def download_rcsb_pdb(pdbid: str, out_dir: str) -> str:
    pdbid = pdbid.strip().upper()
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{pdbid}.pdb")
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
        return out_path
    url = f"https://files.rcsb.org/download/{pdbid}.pdb"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        f.write(r.content)
    return out_path

def pseudocolor(minval, maxval, val):
    minval, maxval = float(minval), float(maxval)
    if maxval == minval:
        h = 120.0
    else:
        h = (float(maxval - val) / (maxval - minval)) * 120.0
    r, g, b = colorsys.hsv_to_rgb(h/360.0, 1.0, 1.0)
    rgb = tuple(int(255*x) for x in (r, g, b))
    return "0x%02x%02x%02x" % rgb

def remap_b2b_colors(df):
    BDcolor, EFcolor, DOcolor = {}, {}, {}
    seqpos = 0
    min_BD, max_BD = float(df["backbone"].min()), float(df["backbone"].max())
    min_DO, max_DO = float(df["disoMine"].min()), float(df["disoMine"].max())
    min_EF, max_EF = float(df["earlyFolding"].min()), float(df["earlyFolding"].max())
    for _, row in df.iterrows():
        seqpos += 1
        BDcolor[seqpos] = pseudocolor(min_BD, max_BD, float(row["backbone"]))
        DOcolor[seqpos] = pseudocolor(min_DO, max_DO, float(row["disoMine"]))
        EFcolor[seqpos] = pseudocolor(min_EF, max_EF, float(row["earlyFolding"]))
    return BDcolor, EFcolor, DOcolor

def display_b2b_4panel_py3dmol(dynamic_properties_df, pdb_path: str, chain=None, ptm_positions=None):
    BDcolor, EFcolor, DOcolor = remap_b2b_colors(dynamic_properties_df)
    with open(pdb_path, "r") as f:
        pdb_txt = f.read()

    view = py3Dmol.view(viewergrid=(2, 2), linked=True, width=950, height=740)
    view.addModel(pdb_txt, "pdb")

    view.setStyle({}, {"cartoon": {"colorscheme": {"prop": "b", "gradient": "rwb", "min": 0.0, "max": 100.0}}}, viewer=(0, 0))
    view.setStyle({}, {"cartoon": {"colorscheme": {"prop": "resi", "map": BDcolor}}}, viewer=(0, 1))
    view.setStyle({}, {"cartoon": {"colorscheme": {"prop": "resi", "map": DOcolor}}}, viewer=(1, 0))
    view.setStyle({}, {"cartoon": {"colorscheme": {"prop": "resi", "map": EFcolor}}}, viewer=(1, 1))

    for panel in [(0,0), (0,1), (1,0), (1,1)]:
        view.setBackgroundColor("white", viewer=panel)

    view.zoomTo()
    view.render()
    return view

def fetch_seq(_):
    try:
        with out_b2b:
            clear_output()
            if _need_acc(out_b2b):
                return
            print("Fetching sequence...")
        _pid, seq = fetch_sequence_aminoacids(state["acc"])
        state["sequence"] = seq
        with out_b2b:
            print(f"✅ Sequence length: {len(seq)} aa")
    except Exception as e:
        _err(out_b2b, e)

def run_b2b(_):
    try:
        if not state.get("sequence"):
            with out_b2b:
                clear_output()
                print("Fetch sequence first.")
            return

        btn_run_b2b.disabled = True
        with out_b2b:
            clear_output()
            print("Running Bio2Byte predictions (this can take a bit)...")

        pred = predict_biophysical_features(state["acc"], state["sequence"])
        state["bio2byte_raw"] = pred
        prot = pred.get("proteins", {}).get(state["acc"], {})
        dyn = pd.DataFrame(prot)

        num_cols = [c for c in dyn.columns if dyn[c].dtype.kind in "if"]
        b2b_metric.options = num_cols
        if num_cols:
            b2b_metric.value = num_cols[0]

        state["dynamic_properties"] = dyn

        with out_b2b:
            print("✅ Done.")
            display(dyn.head())

    except Exception as e:
        _err(out_b2b, e)
    finally:
        btn_run_b2b.disabled = False

def show_b2b(_):
    with out_b2b:
        clear_output()
        if state.get("dynamic_properties") is None:
            print("Run predictions first.")
            return
        display_scrollable_df(state["dynamic_properties"], max_height="420px", max_width="95vw")

def _pdb_with_bfactor_from_df(pdb_in: str, df, value_col: str, out_dir: str, chain: str = None, uni_range=None) -> str:
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"b2b_{value_col}.pdb")

    values = df[value_col].tolist()
    pos_to_val = {i + 1: float(v) for i, v in enumerate(values) if v is not None and v == v}

    u0, u1 = None, None
    if uni_range and len(uni_range) == 2:
        u0, u1 = int(uni_range[0]), int(uni_range[1])

    def _set_b(line, b):
        b_str = f"{b:6.2f}"
        return line[:60] + b_str + line[66:]

    with open(pdb_in, "r") as fin, open(out_path, "w") as fout:
        for line in fin:
            if not (line.startswith("ATOM") or line.startswith("HETATM")):
                fout.write(line)
                continue

            ch = line[21].strip()
            if chain and ch and ch != chain:
                fout.write(line)
                continue

            try:
                resseq = int(line[22:26].strip())
            except:
                fout.write(line)
                continue

            if u0 is not None and not (u0 <= resseq <= u1):
                fout.write(line)
                continue

            if resseq in pos_to_val:
                fout.write(_set_b(line, pos_to_val[resseq]))
            else:
                fout.write(line)

    return out_path

def display_b2b_3D_ngl(dyn_df, pdb_path: str, value_col: str, chain: str = None, uni_range=None):
    out_pdb = _pdb_with_bfactor_from_df(
        pdb_in=pdb_path,
        df=dyn_df,
        value_col=value_col,
        out_dir=state["workdir"],
        chain=(chain or None),
        uni_range=uni_range
    )

    v = nv.show_file(out_pdb)
    v.clear_representations()
    v.add_representation("cartoon", selection="protein", color="grey")

    sel = "protein"
    ch = (chain or "").strip()
    if ch:
        sel = f"protein and :{ch}"
    if uni_range and len(uni_range) == 2:
        u0, u1 = int(uni_range[0]), int(uni_range[1])
        if ch:
            sel = f"{u0}-{u1}:{ch}"
        else:
            sel = f"{u0}-{u1}"

    v.add_representation("cartoon", selection=sel, color_scheme="bfactor")
    try:
        v.center()
    except Exception:
        try:
            v._remote_call("autoView", target="stage")
        except Exception:
            pass
    return v

def show_b2b3d(_):
    try:
        with out_b2b:
            clear_output()

            if state.get("dynamic_properties") is None:
                print("Run predictions first.")
                return

            if not b2b_metric.value:
                print("Pick a metric.")
                return

            pdb_path = None
            chain = None
            uni_range = None

            if b2b_source.value == "af":
                if not state.get("af_path"):
                    print("Fetch AlphaFold model first (Tab 3).")
                    return
                pdb_path = state["af_path"]
                chain = "A"
                uni_range = None
            else:
                pdbid = (b2b_pdb_text.value or "").strip().upper() or (b2b_pdb_dropdown.value or "").strip().upper()
                if not pdbid:
                    print("Provide a PDB ID (type it or pick from the UniProt PDB dropdown).")
                    return
                pdb_path = download_rcsb_pdb(pdbid, out_dir=state["workdir"])

                chain = (b2b_chain_text.value or "").strip().upper() or None

                v = (b2b_chain_dropdown.value or "")
                if "|" in v:
                    parts = (v.split("|") + ["", ""])[:3]
                    _ch, u0, u1 = parts[0], parts[1], parts[2]
                    _ch = (_ch or "").strip().upper()[:1]
                    if _ch and not chain:
                        chain = _ch
                    if u0 and u1:
                        try:
                            uni_range = (int(u0), int(u1))
                        except Exception:
                            uni_range = None

            if viewer_mode.value == "ngl":
                v = display_b2b_3D_ngl(
                    state["dynamic_properties"],
                    pdb_path=pdb_path,
                    value_col=b2b_metric.value,
                    chain=chain,
                    uni_range=uni_range
                )
                display(v)
                return

            # 4-panel py3Dmol
            view = display_b2b_4panel_py3dmol(
                state["dynamic_properties"],
                pdb_path=pdb_path,
                chain=chain
            )
            display(view)

    except Exception as e:
        _err(out_b2b, e)

import threading

# --- Auto-refresh controller for Tab 4 ---
state["b2b_autorefresh"] = True
_b2b_refresh_timer = None

def _b2b_autorefresh(_=None, delay=0.15):
    """Debounced auto refresh for Bio2Byte 3D panel."""
    global _b2b_refresh_timer
    if not state.get("b2b_autorefresh", True):
        return
    # Only auto-refresh if user already has predictions
    if state.get("dynamic_properties") is None:
        return

    # cancel pending
    try:
        if _b2b_refresh_timer is not None:
            _b2b_refresh_timer.cancel()
    except Exception:
        pass

    def _go():
        try:
            show_b2b3d(None)
        except Exception:
            pass

    _b2b_refresh_timer = threading.Timer(delay, _go)
    _b2b_refresh_timer.start()


btn_fetch_seq.on_click(fetch_seq)
btn_run_b2b.on_click(run_b2b)
btn_show_b2b.on_click(show_b2b)
btn_show_b2b3d.on_click(show_b2b3d)

# Auto-refresh whenever these change
viewer_mode.observe(_b2b_autorefresh, names="value")
b2b_metric.observe(_b2b_autorefresh, names="value")
b2b_source.observe(_b2b_autorefresh, names="value")

b2b_pdb_text.observe(_b2b_autorefresh, names="value")
b2b_chain_text.observe(_b2b_autorefresh, names="value")
b2b_pdb_dropdown.observe(_b2b_autorefresh, names="value")
b2b_chain_dropdown.observe(_b2b_autorefresh, names="value")


tab4_controls_pdb = w.VBox([
    w.HBox([b2b_pdb_text, b2b_chain_text]),
    w.HBox([b2b_pdb_dropdown, b2b_chain_dropdown]),
])
tab4_controls_pdb.layout.display = "none"

def _b2b_set_source_ui(*_):
    with out_b2b:
        clear_output()
        if b2b_source.value == "af":
            b2b_pdb_text.value = ""
            b2b_chain_text.value = ""
            try: b2b_pdb_dropdown.value = ""
            except Exception: pass
            try: b2b_chain_dropdown.value = ""
            except Exception: pass
            tab4_controls_pdb.layout.display = "none"
            print("AlphaFold selected. Use Tab 3 to fetch the AlphaFold model, then come back here to color by Bio2Byte properties.")
        else:
            tab4_controls_pdb.layout.display = ""
            print("PDB selected. Provide a PDB ID (or pick from UniProt PDB dropdown). Optional: choose chain/range from dropdown to restrict coloring.")

b2b_source.observe(lambda ch: _b2b_set_source_ui(), names="value")
_b2b_set_source_ui()

tab4 = w.VBox([
    w.HTML("<h3>Bio2Byte biophysical predictions</h3>"),
    w.HBox([btn_fetch_seq, btn_run_b2b, btn_show_b2b, btn_show_b2b3d]),
    w.HBox([b2b_source, viewer_mode, b2b_metric]),
    tab4_controls_pdb,
    out_b2b
])

# Initialize Tab 4 dropdowns (safe)
try:
    _refresh_b2b_pdb_dropdowns()
except Exception:
    pass



# ----------------------------
# Shared UniProt PDB dropdown helpers for Tabs 5/6
# ----------------------------
def _pdb_options_from_uniprot_refs():
    refs = state.get("uniprot_pdb_refs") or []
    pdb_ids = sorted({(r.get("pdb_id") or "").upper() for r in refs if r.get("pdb_id")})
    return [("", "")] + [(pid, pid) for pid in pdb_ids]

def _chain_options_for_uniprot_pdb(pdb_id: str):
    pdb_id = (pdb_id or "").upper().strip()
    refs = state.get("uniprot_pdb_refs") or []
    cr = {}
    for r in refs:
        if (r.get("pdb_id") or "").upper() != pdb_id:
            continue
        for ch, rng in (r.get("chain_ranges") or {}).items():
            cr[ch] = rng
    opts = [("", "")]
    for ch in sorted(cr.keys()):
        rng = cr[ch]
        if rng and len(rng) == 2:
            opts.append((f"{ch} ({rng[0]}-{rng[1]})", ch))
        else:
            opts.append((f"{ch} (range n/a)", ch))
    return opts

def _selected_uniprot_chain_range(pdb_id: str, chain_id: str):
    pdb_id = (pdb_id or "").upper().strip()
    chain_id = (chain_id or "").upper().strip()[:1]
    if not pdb_id or not chain_id:
        return None
    for r in state.get("uniprot_pdb_refs") or []:
        if (r.get("pdb_id") or "").upper() != pdb_id:
            continue
        rng = (r.get("chain_ranges") or {}).get(chain_id)
        if rng and len(rng) == 2:
            return int(rng[0]), int(rng[1])
    return None

def _set_dd_options(dd, options):
    values = [v for _, v in options]
    old = dd.value
    dd.options = options
    dd.value = old if old in values else ""

def _sync_chain_dropdown_for_pdb(pdb_dd_widget, chain_dd_widget, range_label_widget=None):
    pid = (pdb_dd_widget.value or "").upper().strip()
    if not pid:
        chain_dd_widget.options = [("", "")]
        chain_dd_widget.value = ""
        if range_label_widget is not None:
            range_label_widget.value = ""
        return
    opts = _chain_options_for_uniprot_pdb(pid)
    _set_dd_options(chain_dd_widget, opts)
    if range_label_widget is not None:
        ch = (chain_dd_widget.value or "").upper().strip()[:1]
        rng = _selected_uniprot_chain_range(pid, ch)
        if ch and rng:
            range_label_widget.value = f"<b>UniProt range:</b> {ch} = {rng[0]}–{rng[1]}"
        elif ch:
            range_label_widget.value = f"<b>UniProt range:</b> {ch} = n/a"
        else:
            range_label_widget.value = ""

def _extract_chain_or_range(pdb_path: str, chain_id: str, out_path: str, start=None, end=None) -> str:
    chain_id = (chain_id or "A").strip()[:1]
    if start is None or end is None:
        start, end = chain_range_from_pdb(pdb_path, chain_id)
    parser = PDBParser(QUIET=True)
    io = PDBIO()
    io.set_structure(parser.get_structure("X", pdb_path))
    io.save(out_path, select=ChainRangeSelect(chain_id, int(start), int(end)))
    return out_path

# ----------------------------
# TAB 5: RIN (Residue Interaction Network)
# ----------------------------
out_rin = w.Output(layout={"border":"1px solid #ddd","padding":"6px"})
rin_source = w.RadioButtons(
    options=[("AlphaFold", "alphafold"), ("UniProt PDB dropdown", "uniprot_pdb"), ("Upload PDB", "upload")],
    value="alphafold",
    description="Structure:",
    layout=w.Layout(width="260px")
)
btn_dl_af = w.Button(description="Download AlphaFold PDB", button_style="warning")
rin_pdb_dd = w.Dropdown(options=[("", "")], value="", description="UniProt PDB:", layout=w.Layout(width="420px"))
rin_chain_dd = w.Dropdown(options=[("", "")], value="", description="Chain/range:", layout=w.Layout(width="420px"))
rin_range_lbl = w.HTML("")
upload_pdb = w.FileUpload(accept=".pdb", multiple=False, description="Upload PDB")
chain_rin = w.Text(description="Upload/AF chain:", placeholder="A", value="A", layout=w.Layout(width="220px"))
cutoff = w.FloatSlider(description="Cutoff Å:", min=4.0, max=12.0, step=0.5, value=8.0, readout=True)
btn_build_rin = w.Button(description="Build RIN", button_style="danger")
btn_show_rin = w.Button(description="Show RIN", button_style="success")

import os

def _save_upload_to_path(upl: w.FileUpload, out_dir: str) -> str:
    if not upl.value:
        raise ValueError("No PDB uploaded")
    v = upl.value
    if isinstance(v, dict):
        fname, item = next(iter(v.items()))
        name = item.get("metadata", {}).get("name") or item.get("name") or fname or "uploaded.pdb"
        content = item.get("content", b"")
    elif isinstance(v, (list, tuple)):
        item = v[0]
        name = item.get("metadata", {}).get("name") or item.get("name") or "uploaded.pdb"
        content = item.get("content", b"")
    else:
        raise TypeError(f"Unexpected FileUpload.value type: {type(v)}")
    if isinstance(content, memoryview):
        content = content.tobytes()
    elif isinstance(content, bytearray):
        content = bytes(content)
    os.makedirs(out_dir, exist_ok=True)
    fp = os.path.join(out_dir, name)
    with open(fp, "wb") as f:
        f.write(content)
    if os.path.getsize(fp) < 100:
        raise ValueError(f"Uploaded file looks too small ({os.path.getsize(fp)} bytes). Not a valid PDB?")
    return fp

def _refresh_rin_pdb_dropdowns():
    _set_dd_options(rin_pdb_dd, _pdb_options_from_uniprot_refs())
    _sync_chain_dropdown_for_pdb(rin_pdb_dd, rin_chain_dd, rin_range_lbl)

def _update_rin_chain_dropdown(*_):
    _sync_chain_dropdown_for_pdb(rin_pdb_dd, rin_chain_dd, rin_range_lbl)

def _update_rin_source_ui(*_):
    if rin_source.value == "uniprot_pdb":
        rin_uniprot_controls.layout.display = ""
        rin_upload_controls.layout.display = "none"
        btn_dl_af.layout.display = "none"
        chain_rin.description = "Fallback chain:"
    elif rin_source.value == "upload":
        rin_uniprot_controls.layout.display = "none"
        rin_upload_controls.layout.display = ""
        btn_dl_af.layout.display = "none"
        chain_rin.description = "Upload chain:"
    else:
        rin_uniprot_controls.layout.display = "none"
        rin_upload_controls.layout.display = "none"
        btn_dl_af.layout.display = ""
        chain_rin.description = "AF chain:"

rin_pdb_dd.observe(_update_rin_chain_dropdown, names="value")
rin_chain_dd.observe(_update_rin_chain_dropdown, names="value")
rin_source.observe(_update_rin_source_ui, names="value")

def dl_af(_):
    try:
        with out_rin:
            clear_output()
            if _need_acc(out_rin): 
                return
            print("Downloading AlphaFold PDB...")
        fp = download_alphafold_pdb(state["acc"], state["workdir"])
        state["rin_pdb_path"] = fp
        with out_rin:
            print(f"✅ Saved: {fp}")
    except Exception as e:
        _err(out_rin, e)

def _resolve_rin_structure():
    src = rin_source.value
    if src == "upload":
        pdb_path = _save_upload_to_path(upload_pdb, state["workdir"])
        ch = (chain_rin.value or "A").strip()[:1]
        rng = None
        label = f"uploaded {os.path.basename(pdb_path)}"
    elif src == "uniprot_pdb":
        pid = (rin_pdb_dd.value or "").upper().strip()
        if not pid:
            raise ValueError("Select a UniProt PDB entry first.")
        pdb_path = _download_pdb_to_workdir(pid, state["workdir"])
        ch = (rin_chain_dd.value or chain_rin.value or "A").strip()[:1]
        rng = _selected_uniprot_chain_range(pid, ch)
        label = f"{pid} chain {ch}"
    else:
        if state.get("rin_pdb_path") and os.path.exists(state["rin_pdb_path"]):
            pdb_path = state["rin_pdb_path"]
        else:
            pdb_path = download_alphafold_pdb(state["acc"], state["workdir"])
            state["rin_pdb_path"] = pdb_path
        ch = (chain_rin.value or "A").strip()[:1]
        rng = None
        label = f"AlphaFold chain {ch}"
    return pdb_path, ch, rng, label

def build_rin(_):
    try:
        with out_rin:
            clear_output()
            if _need_acc(out_rin):
                return
            pdb_path, ch, rng, label = _resolve_rin_structure()
            print("Using pdb_path:", pdb_path)
            print("File size (bytes):", os.path.getsize(pdb_path))
            if rng:
                print(f"Building RIN from {label}, UniProt mapped range {rng[0]}-{rng[1]}, cutoff {cutoff.value} Å ...")
                rin_input = os.path.join(state["workdir"], f"rin_{os.path.basename(pdb_path).replace('.pdb','')}_{ch}_{rng[0]}_{rng[1]}.pdb")
                _extract_chain_or_range(pdb_path, ch, rin_input, rng[0], rng[1])
            else:
                print(f"Building RIN from {label}, cutoff {cutoff.value} Å ...")
                rin_input = pdb_path

        tmp = build_geometry_graph_from_pdb(rin_input, chain=ch, cutoff=float(cutoff.value))
        G = tmp[0] if isinstance(tmp, tuple) else tmp
        print("Chain used for RIN:", G.graph.get("chain"))
        ptm_pos = None
        mut_pos = None
        if state.get("ptm_table") is not None and "position" in state["ptm_table"].columns:
            ptm_pos = list(set(state["ptm_table"]["position"].dropna().astype(int).tolist()))
        if state.get("variants_df") is not None and "position" in state["variants_df"].columns:
            mut_pos = list(set(state["variants_df"]["position"].dropna().astype(int).tolist()))

        out_html = os.path.join(state["workdir"], f"rin_pyvis_{state['acc'] or 'session'}.html")
        html_path = nx_rin_to_pyvis_default(G, ptm_positions=ptm_pos, mutation_positions=mut_pos, out_html=out_html)
        state["rin_html"] = html_path

        with out_rin:
            print("✅ RIN built.")
            print("Click 'Show RIN' to open the interactive network.")

    except Exception as e:
        _err(out_rin, e)

from IPython.display import HTML

def show_rin(_):
    with out_rin:
        clear_output()
        p = state.get("rin_html")
        if not p or not os.path.exists(p):
            print("No RIN HTML yet. Build it first.")
            return
        with open(p, "r", encoding="utf-8") as f:
            html = f.read()
        display(HTML(html))

btn_dl_af.on_click(dl_af)
btn_build_rin.on_click(build_rin)
btn_show_rin.on_click(show_rin)

rin_uniprot_controls = w.VBox([w.HBox([rin_pdb_dd, rin_chain_dd]), rin_range_lbl])
rin_upload_controls = w.HBox([upload_pdb])
_update_rin_source_ui()
try:
    _refresh_rin_pdb_dropdowns()
except Exception:
    pass

tab5 = w.VBox([
    w.HTML("<h3>Residue Interaction Network (PyVis)</h3>"),
    w.HBox([rin_source, btn_dl_af]),
    rin_uniprot_controls,
    rin_upload_controls,
    w.HBox([chain_rin, cutoff]),
    w.HBox([btn_build_rin, btn_show_rin]),
    out_rin
])

# ----------------------------
# TAB 6: TM-align tool with upload or UniProt PDB dropdowns
# ----------------------------
tm_source1 = w.RadioButtons(
    options=[("AlphaFold", "alphafold"), ("UniProt PDB dropdown", "uniprot_pdb"), ("Upload PDB", "upload")],
    value="uniprot_pdb",
    description="Structure 1:",
    layout=w.Layout(width="260px")
)
tm_source2 = w.RadioButtons(
    options=[("AlphaFold", "alphafold"), ("UniProt PDB dropdown", "uniprot_pdb"), ("Upload PDB", "upload")],
    value="uniprot_pdb",
    description="Structure 2:",
    layout=w.Layout(width="260px")
)
tm_pdb1_dd = w.Dropdown(options=[("", "")], value="", description="PDB 1:", layout=w.Layout(width="390px"))
tm_pdb2_dd = w.Dropdown(options=[("", "")], value="", description="PDB 2:", layout=w.Layout(width="390px"))
tm_chain1_dd = w.Dropdown(options=[("", "")], value="", description="Chain 1:", layout=w.Layout(width="390px"))
tm_chain2_dd = w.Dropdown(options=[("", "")], value="", description="Chain 2:", layout=w.Layout(width="390px"))
tm_range1_lbl = w.HTML("")
tm_range2_lbl = w.HTML("")

# Reuse the original upload/range widgets, but relabel the upload controls.
upload1.description = "Upload PDB 1"
upload2.description = "Upload PDB 2"
chain1.description = "Chain 1"
chain2.description = "Chain 2"

out_tm = out
workdir = state.get("workdir") or workdir

def _refresh_tm_pdb_dropdowns():
    opts = _pdb_options_from_uniprot_refs()
    _set_dd_options(tm_pdb1_dd, opts)
    _set_dd_options(tm_pdb2_dd, opts)
    _sync_chain_dropdown_for_pdb(tm_pdb1_dd, tm_chain1_dd, tm_range1_lbl)
    _sync_chain_dropdown_for_pdb(tm_pdb2_dd, tm_chain2_dd, tm_range2_lbl)

def _update_tm_chain1(*_):
    _sync_chain_dropdown_for_pdb(tm_pdb1_dd, tm_chain1_dd, tm_range1_lbl)
    if tm_chain1_dd.value:
        chain1.value = tm_chain1_dd.value
        rng = _selected_uniprot_chain_range(tm_pdb1_dd.value, tm_chain1_dd.value)
        if rng:
            start1.value, end1.value = rng[0], rng[1]

def _update_tm_chain2(*_):
    _sync_chain_dropdown_for_pdb(tm_pdb2_dd, tm_chain2_dd, tm_range2_lbl)
    if tm_chain2_dd.value:
        chain2.value = tm_chain2_dd.value
        rng = _selected_uniprot_chain_range(tm_pdb2_dd.value, tm_chain2_dd.value)
        if rng:
            start2.value, end2.value = rng[0], rng[1]

def _update_tm_source_ui(*_):
    tm1_uniprot_controls.layout.display = "" if tm_source1.value == "uniprot_pdb" else "none"
    tm1_upload_controls.layout.display = "" if tm_source1.value == "upload" else "none"
    tm2_uniprot_controls.layout.display = "" if tm_source2.value == "uniprot_pdb" else "none"
    tm2_upload_controls.layout.display = "" if tm_source2.value == "upload" else "none"

    # AlphaFold uses chain A by default and the full model range is auto-filled at run time.
    if tm_source1.value == "alphafold":
        chain1.value = "A"
    if tm_source2.value == "alphafold":
        chain2.value = "A"

tm_pdb1_dd.observe(_update_tm_chain1, names="value")
tm_chain1_dd.observe(_update_tm_chain1, names="value")
tm_pdb2_dd.observe(_update_tm_chain2, names="value")
tm_chain2_dd.observe(_update_tm_chain2, names="value")
tm_source1.observe(_update_tm_source_ui, names="value")
tm_source2.observe(_update_tm_source_ui, names="value")

def _resolve_tm_structure(which: int):
    source = tm_source1.value if which == 1 else tm_source2.value

    if source == "alphafold":
        if not state.get("acc"):
            raise ValueError("Set a UniProt accession first before using AlphaFold in TM-align.")

        # Reuse the AlphaFold model fetched in Tab 3 when available.
        # If it has not been fetched yet, download it here so Tab 6 works independently.
        pdb = state.get("af_path")
        if not pdb or not os.path.exists(pdb):
            pdb = download_alphafold_pdb(state["acc"], out_dir=state["workdir"])
            state["af_path"] = pdb

        ch = (chain1.value if which == 1 else chain2.value) or "A"
        ch = ch.strip()[:1] or "A"
        s0, e0 = chain_range_from_pdb(pdb, ch)

        if which == 1:
            chain1.value, start1.value, end1.value = ch, int(s0), int(e0)
        else:
            chain2.value, start2.value, end2.value = ch, int(s0), int(e0)

        label = f"AlphaFold_{state['acc']}_{ch}"
        return pdb, ch, int(s0), int(e0), label

    if which == 1:
        if source == "upload":
            pdb = save_upload(upload1, workdir)
            ch = (chain1.value or "A").strip()[:1]
            s0, e0 = int(start1.value), int(end1.value)
            label = os.path.basename(pdb)
        else:
            pid = (tm_pdb1_dd.value or "").upper().strip()
            if not pid:
                raise ValueError("Select PDB 1 from the UniProt dropdown, switch Structure 1 to AlphaFold, or upload a PDB.")
            pdb = _download_pdb_to_workdir(pid, workdir)
            ch = (tm_chain1_dd.value or chain1.value or "A").strip()[:1]
            rng = _selected_uniprot_chain_range(pid, ch)
            s0, e0 = rng if rng else chain_range_from_pdb(pdb, ch)
            chain1.value, start1.value, end1.value = ch, int(s0), int(e0)
            label = f"{pid}_{ch}"
    else:
        if source == "upload":
            pdb = save_upload(upload2, workdir)
            ch = (chain2.value or "A").strip()[:1]
            s0, e0 = int(start2.value), int(end2.value)
            label = os.path.basename(pdb)
        else:
            pid = (tm_pdb2_dd.value or "").upper().strip()
            if not pid:
                raise ValueError("Select PDB 2 from the UniProt dropdown, switch Structure 2 to AlphaFold, or upload a PDB.")
            pdb = _download_pdb_to_workdir(pid, workdir)
            ch = (tm_chain2_dd.value or chain2.value or "A").strip()[:1]
            rng = _selected_uniprot_chain_range(pid, ch)
            s0, e0 = rng if rng else chain_range_from_pdb(pdb, ch)
            chain2.value, start2.value, end2.value = ch, int(s0), int(e0)
            label = f"{pid}_{ch}"
    return pdb, ch, int(s0), int(e0), label

def autofill_ranges(_):
    out.clear_output()
    with out:
        try:
            pdb1, ch1, s1, e1, label1 = _resolve_tm_structure(1)
            pdb2, ch2, s2, e2, label2 = _resolve_tm_structure(2)
            start1.value, end1.value = s1, e1
            start2.value, end2.value = s2, e2
            print(f"Ranges auto-filled: {label1} {ch1}:{s1}-{e1}; {label2} {ch2}:{s2}-{e2}")
        except Exception as e:
            print("ERROR:", e)

btn_range.on_click(autofill_ranges)

def run_align(_):
    out.clear_output()
    with out:
        try:
            pdb1, ch1, s1, e1, label1 = _resolve_tm_structure(1)
            pdb2, ch2, s2, e2, label2 = _resolve_tm_structure(2)

            seg1 = os.path.join(workdir, "seg1.pdb")
            seg2 = os.path.join(workdir, "seg2.pdb")

            _extract_chain_or_range(pdb1, ch1, seg1, s1, e1)
            _extract_chain_or_range(pdb2, ch2, seg2, s2, e2)

            aligned_pdb, stdout = run_tmalign_write(seg1, seg2, out_dir=workdir, out_name="aligned")

            print(f"Aligned {label1} chain {ch1}:{s1}-{e1} against {label2} chain {ch2}:{s2}-{e2}")
            print(stdout.splitlines()[0] if stdout.splitlines() else "TM-align finished.")
            display(visualize_ngl(seg2, aligned_pdb))

        except Exception as e:
            print("ERROR:", e)

btn_run.on_click(run_align)

tm1_uniprot_controls = w.VBox([w.HBox([tm_pdb1_dd, tm_chain1_dd]), tm_range1_lbl])
tm2_uniprot_controls = w.VBox([w.HBox([tm_pdb2_dd, tm_chain2_dd]), tm_range2_lbl])
tm1_upload_controls = w.HBox([upload1])
tm2_upload_controls = w.HBox([upload2])
_update_tm_source_ui()
try:
    _refresh_tm_pdb_dropdowns()
except Exception:
    pass

tmalign_ui = widgets.VBox([
    widgets.HBox([tm_source1, tm_source2]),
    widgets.HTML("<b>Structure 1</b>"),
    tm1_uniprot_controls,
    tm1_upload_controls,
    widgets.HBox([chain1, start1, end1]),
    widgets.HTML("<b>Structure 2</b>"),
    tm2_uniprot_controls,
    tm2_upload_controls,
    widgets.HBox([chain2, start2, end2]),
    widgets.HBox([btn_range, btn_run]),
    out,
])

tab6 = w.VBox([
    w.HTML("<h3>TM-align (AlphaFold, UniProt PDB dropdown, or uploaded structures)</h3>"),
    w.HTML("<p><b>Note:</b> TM-align must be installed in the container/server and available on PATH as <code>TM-align</code>.</p>"),
    tmalign_ui,
])

# ----------------------------
# Assemble tabs
# ----------------------------
tabs = w.Tab(children=[tab1, tab2, tab3, tab4, tab5, tab6])
titles = ["1) PTMs", "2) Variants", "3) 3D Viewer", "4) Bio2Byte", "5) RIN", "6) TM-align"]
for i, t in enumerate(titles):
    tabs.set_title(i, t)

display(w.VBox([top, tabs]))

